# Benchmark dei 15 modelli su telemetria SolarTech Lab

Estensione di `solar_telemetry.ipynb`. Prende le **stesse finestre** di telemetria misurata e le
manda a tutti i checkpoint del sweep piu' al Chronos-Bolt pubblicato, poi riporta MAE, bias,
varianza dell'errore e weighted quantile loss in una tabella sola.

### Cosa misura, e cosa non misura

Il baseline della tabella e' `amazon/chronos-bolt-tiny` **come rilasciato**. Va letto sapendo cosa
separa quel checkpoint dagli altri quindici: e' preaddestrato su un corpus osservativo di ordine
$10^7$ serie *che comprende energia e meteo*, mentre i quindici hanno visto solo la mixture
sintetica del sweep. Il divario verso il baseline e' quindi **differenza di corpus di
addestramento**, non differenza di geometria, e non e' un risultato sull'aliasing.

Per questo ogni riga porta **due** colonne di scarto:

| colonna | confronto | cosa isola |
|---|---|---|
| `d_MAE_vs_base` | verso `chronos-bolt-tiny` pubblicato | corpus **e** geometria insieme |
| `d_MAE_vs_1616` | verso `p16-s16-seed42` retrainato | **solo** la geometria, a parita' di corpus, step e seed |

La seconda e' l'unica delle due che isola la variabile del progetto: `p16-s16` e' la geometria
stock di Bolt, addestrata esattamente come le altre quattordici. Se in discussione qualcuno chiede
"e a parita' di dati?", la risposta e' quella colonna.

### Il canale

Si usa **`G_h`**, l'irradianza orizzontale, non `PV_Power`. La ragione e' aritmetica: `PV_Power` e'
`NaN` fuori dalle ore di luce e il suo blocco contiguo piu' lungo in tutto il file e' di **880
campioni**, contro i $2048 + 64 = 2112$ che servono per una finestra a contesto pieno. Su
`PV_Power` il numero di finestre utilizzabili a questo contesto e' **zero**. `G_h` e' valido al
99,1%, e' il driver fisico della potenza, e lascia il contesto alla lunghezza su cui i modelli sono
stati addestrati.

## 0, Configurazione

In [7]:
# --------------------------------------------------------------------------------------- #
#  CONFIGURAZIONE
# --------------------------------------------------------------------------------------- #
CSV_PATH  = "../data/dataset/Dataset-SolarTechLab.csv"   # se non esiste viene cercato altrove
CANALE    = "G_h"            # G_h | G_tilt | T_air | W_s   (PV_Power non regge il contesto 2048)

CTX, PRED = 2048, 64         # le lunghezze del training del sweep (train_sweep.py)
L_FIN     = CTX + PRED       # 2112 campioni contigui per finestra

N_FINESTRE = 150             # finestre DISGIUNTE. Piu' basso e' il numero, piu' margine ha
                             # ciascun blocco per centrare l'ora del taglio: a 150 il margine
                             # e' di circa 23 ore e la copertura oraria e' quasi esatta.
                             # Il massimo che il file concede a questo contesto e' ~207.
SEED       = 42
N_BOOT     = 2000            # ricampionamenti per gli intervalli sugli scarti appaiati

BASE_ID     = "amazon/chronos-bolt-tiny"   # il baseline della tabella, come rilasciato
RIF_INTERNO = (16, 16)                     # il riferimento a parita' di corpus, retrainato

BATCH   = 64
DEVICE  = None               # None = cuda se c'e'
OUT_DIR = "_run/solar_bench"

# Predittore finto al posto dei modelli: serve a provare campionamento, metriche e bootstrap
# senza GPU e senza checkpoint. NON produce risultati, e la tabella lo dichiara.
MODO_PROVA = False           #@param {type:"boolean"}

# Soglie di regime, in gradi di elevazione solare, per la ripartizione in coda al notebook.
ELEV_GIORNO = 10.0
SITE_LAT, SITE_LON, SITE_TZ = 45.502, 9.156, 1.0     # SolarTech Lab, Politecnico di Milano

## 0b, L'ambiente

Il notebook ha bisogno di tre cose che stanno tutte **nel repo**: il CSV della telemetria,
`probe_lib.py` per l'elenco delle geometrie e `model_loader.py` per risolvere i checkpoint.
Se il kernel gira dentro il repo non serve nulla e questa cella non fa niente.

Se il kernel gira altrove la cella ce li porta. Il caso da tenere a mente e' un **kernel Colab
pilotato da VS Code**: il notebook sta sul disco locale, il codice esegue su `/content`, e sono
due filesystem diversi. Una cartella creata in locale il kernel non la vede, e un percorso
relativo si risolve rispetto alla working directory del **kernel**, non a quella del notebook.
Da qui l'errore su `../data/dataset/...`, e subito dopo quello sugli import.

In [8]:
#@title 0b. Drive, repo, percorsi e dipendenze  { display-mode: "form" }
MONTA_DRIVE  = True   #@param {type:"boolean"}
CLONA_REPO   = True   #@param {type:"boolean"}
REPO_URL     = "https://github.com/FedericoSabbadini/patchAliasing.git"  #@param {type:"string"}
REPO_REF     = "main"  #@param {type:"string"}
GITHUB_TOKEN = ""     #@param {type:"string"}
CLONE_DIR    = "/content/patchAliasing"  #@param {type:"string"}

import os, sys, shutil, subprocess, importlib, importlib.util
from pathlib import Path

# ---------------------------------------------------------------- Drive
# Serve solo se il CSV sta li'. Con un kernel Colab pilotato da VS Code il mount puo' non
# riuscire, perche' l'autenticazione passa dal frontend di Colab: in quel caso non e' un
# errore fatale, si prosegue e il CSV si cerca altrove.
DRIVE = None
if MONTA_DRIVE and not Path("/content/drive/MyDrive").is_dir():
    try:
        from google.colab import drive as _drv
        _drv.mount("/content/drive")
    except Exception as _e:
        print(f"Drive non montato ({type(_e).__name__}: {_e}).")
        print("  Se il CSV sta su Drive, montalo a mano o copia il file sul kernel.")
if Path("/content/drive/MyDrive").is_dir():
    DRIVE = Path("/content/drive/MyDrive"); print(f"Drive: {DRIVE}")

# ---------------------------------------------------------------- repo
# Il notebook importa probe_lib e model_loader e legge il CSV: stanno tutti nel repo. Se il
# kernel non gira dentro il repo - e non ci gira quando il notebook e' locale e il kernel e'
# su Colab - il repo va portato sul kernel, altrimenti falliscono prima la lettura del CSV e
# poi gli import, con due errori che sembrano scollegati.
MARCATORE = Path("chronos/bayesian/probe_lib.py")

def radice_repo():
    if (Path(CLONE_DIR) / MARCATORE).is_file():
        return Path(CLONE_DIR)
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / MARCATORE).is_file():
            return base
    if DRIVE is not None:
        for c in (DRIVE / "patchAliasing", DRIVE / "Colab Notebooks/patchAliasing"):
            if (c / MARCATORE).is_file():
                return c
    return None

RADICE = radice_repo()
if RADICE is None and CLONA_REPO:
    _url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    print(f"repo non trovato sul kernel: clono {REPO_URL} @ {REPO_REF} in {CLONE_DIR}")
    if os.path.isdir(CLONE_DIR):
        shutil.rmtree(CLONE_DIR)
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, _url, CLONE_DIR],
                        capture_output=True, text=True)
    if _r.returncode != 0:
        _err = _r.stderr.strip()
        if GITHUB_TOKEN:
            _err = _err.replace(GITHUB_TOKEN, "***")
        raise RuntimeError("git clone fallito:\n" + _err + "\n\nSe il repository e' privato, "
                           "incolla in GITHUB_TOKEN un token con accesso in lettura.")
    subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", REPO_URL], check=False)
    RADICE = radice_repo()

if RADICE is None:
    raise RuntimeError(
        "Repo non trovato e clone disattivato. Servono CSV, probe_lib.py e model_loader.py sul "
        "KERNEL: con un kernel Colab pilotato da VS Code il notebook e il kernel stanno su due "
        "filesystem diversi, e una cartella creata in locale il kernel non la vede. "
        "Metti CLONA_REPO = True, oppure copia il repo sul kernel.")
print(f"repo: {RADICE}")

# gli import del progetto: la cartella bayesian deve stare nel path del KERNEL
BAY = str((RADICE / "chronos/bayesian").resolve())
if BAY not in sys.path:
    sys.path.insert(0, BAY)

# ---------------------------------------------------------------- il CSV
# Ordine: quello configurato, quello del repo, quello su Drive. Il primo che esiste vince.
_nome = Path(CSV_PATH).name
if not Path(CSV_PATH).is_file():
    _cand = [RADICE / "chronos/data/dataset" / _nome]
    if DRIVE is not None:
        _cand += [DRIVE / _nome, DRIVE / "data/dataset" / _nome,
                  DRIVE / "patchAliasing/chronos/data/dataset" / _nome]
    for _c in _cand:
        if _c.is_file():
            CSV_PATH = str(_c)
            print(f"CSV_PATH corretto in {CSV_PATH}")
            break
if Path(CSV_PATH).is_file():
    print(f"dati: {CSV_PATH}  ({Path(CSV_PATH).stat().st_size/1e6:.1f} MB)")
else:
    print(f"CSV ancora non trovato come {CSV_PATH}: ci riprova la cella 1 con una ricerca piu' larga")

# ---------------------------------------------------------------- dipendenze
if MODO_PROVA:
    print("MODO PROVA: import del progetto e chronos non richiesti")
else:
    if importlib.util.find_spec("torch") is None:
        raise RuntimeError("torch non disponibile sul kernel.")
    if importlib.util.find_spec("chronos") is None:
        print("installo chronos-forecasting ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chronos-forecasting"],
                       check=True)
        importlib.invalidate_caches()
    for _m in ("probe_lib", "model_loader", "chronos"):
        if importlib.util.find_spec(_m) is None:
            raise RuntimeError(f"{_m} non importabile. probe_lib e model_loader sono attesi in {BAY}")
    import torch
    print(f"import del progetto: ok   torch {torch.__version__}   "
          f"cuda {'si' if torch.cuda.is_available() else 'no'}")

Mounted at /content/drive
Drive: /content/drive/MyDrive
repo non trovato sul kernel: clono https://github.com/FedericoSabbadini/patchAliasing.git @ main in /content/patchAliasing
repo: /content/patchAliasing
CSV_PATH corretto in /content/patchAliasing/chronos/data/dataset/Dataset-SolarTechLab.csv
dati: /content/patchAliasing/chronos/data/dataset/Dataset-SolarTechLab.csv  (26.8 MB)
installo chronos-forecasting ...
import del progetto: ok   torch 2.11.0+cpu   cuda no


## 1, Caricamento e pulizia

In [9]:
import json, time, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

def trova_csv(indicato):
    """Il percorso indicato se c'e', altrimenti il file cercato dove finisce di solito.

    Il percorso di default e' relativo a chronos/bayesian/, che e' giusto in locale e sbagliato
    su Colab, dove la working directory e' /content. Invece di far fallire la lettura con un
    percorso che non dice niente, si guarda nei posti plausibili e si stampa quello trovato.
    """
    nome = Path(indicato).name
    if Path(indicato).is_file():
        return Path(indicato)
    candidati = [
        Path.cwd()/nome,
        Path.cwd()/"data/dataset"/nome,
        Path("../data/dataset")/nome, Path("../../data/dataset")/nome,
        Path("/content")/nome,
        Path("/content/data/dataset")/nome,
        Path("/content/patchAliasing/chronos/data/dataset")/nome,
        Path("/content/drive/MyDrive/patchAliasing/chronos/data/dataset")/nome,
        Path("/content/drive/MyDrive/patchAliasing/data/dataset")/nome,
        Path("/content/drive/MyDrive")/nome,
    ]
    for c in candidati:
        if c.is_file():
            print(f"CSV_PATH non esisteva: uso {c}")
            return c
    # ultima risorsa: una ricerca vera, limitata ai posti che su Colab hanno senso
    for radice in (Path.cwd(), Path("/content"), Path("/content/drive/MyDrive")):
        if not radice.is_dir(): continue
        try: trovato = next(radice.rglob(nome), None)
        except (PermissionError, OSError): trovato = None
        if trovato: print(f"CSV_PATH non esisteva: trovato con una ricerca sotto {radice}"); return trovato
    raise FileNotFoundError(
        f"{nome} non trovato. Cercato in: {indicato}, poi " + ", ".join(str(c) for c in candidati)
        + ".\n   Metti il file in una di queste posizioni, oppure imposta CSV_PATH al percorso"
        + " assoluto:\n   CSV_PATH = \"/content/data/dataset/" + nome + "\"\n"
        + "   Su Colab il caricamento diretto si fa con:  from google.colab import files; files.upload()")

CSV_PATH = str(trova_csv(CSV_PATH))
print(f"dati: {CSV_PATH}  ({Path(CSV_PATH).stat().st_size/1e6:.1f} MB)")

RAW = pd.read_csv(CSV_PATH, sep=";", parse_dates=["Time"], dayfirst=True)
RAW = RAW.replace(-999999.0, np.nan)
assert CANALE in RAW.columns, f"{CANALE} non fra {list(RAW.columns)}"

Y_ALL = RAW[CANALE].to_numpy(float)
OK    = np.isfinite(Y_ALL)
T_TOT = len(Y_ALL)
print(f"{T_TOT:,} righe   {RAW.Time.min().date()} -> {RAW.Time.max().date()}")
print(f"{CANALE}: {OK.sum():,} validi ({100*OK.mean():.1f}%)   "
      f"intervallo {np.nanmin(Y_ALL):.4g} .. {np.nanmax(Y_ALL):.4g}")

def _elevazione(ts, lat=SITE_LAT, lon=SITE_LON, tz=SITE_TZ):
    """Elevazione solare in gradi (equazioni NOAA a bassa precisione), come in solar_telemetry."""
    t = pd.DatetimeIndex(ts); utc = t - pd.Timedelta(hours=tz)
    n  = utc.dayofyear.to_numpy(float)
    hr = utc.hour.to_numpy(float) + utc.minute.to_numpy(float)/60 + utc.second.to_numpy(float)/3600
    g  = 2*np.pi/365.0*(n - 1 + (hr - 12)/24.0)
    eq = 229.18*(0.000075 + 0.001868*np.cos(g) - 0.032077*np.sin(g)
                 - 0.014615*np.cos(2*g) - 0.040849*np.sin(2*g))
    dec = (0.006918 - 0.399912*np.cos(g) + 0.070257*np.sin(g) - 0.006758*np.cos(2*g)
           + 0.000907*np.sin(2*g) - 0.002697*np.cos(3*g) + 0.00148*np.sin(3*g))
    tst = (hr*60 + eq + 4*lon) % 1440
    ha  = np.deg2rad(tst/4.0 - 180.0); la = np.deg2rad(lat)
    return np.rad2deg(np.arcsin(np.clip(np.sin(la)*np.sin(dec)
                                        + np.cos(la)*np.cos(dec)*np.cos(ha), -1, 1)))

ELEV = _elevazione(RAW["Time"])

dati: /content/patchAliasing/chronos/data/dataset/Dataset-SolarTechLab.csv  (26.8 MB)
525,601 righe   2017-01-01 -> 2018-01-01
G_h: 520,747 validi (99.1%)   intervallo 0 .. 1312


## 2, Le finestre

Due requisiti, e sono in tensione fra loro.

**Disgiunte.** Due finestre distanti un minuto condividono $2047$ campioni di contesto su $2048$:
campionare minuti a caso produce molte osservazioni e pochissima informazione indipendente. Qui le
finestre non si toccano, quindi la finestra **e'** l'unita' indipendente e il bootstrap piu' avanti
non ha bisogno di blocchi.

**A ore diverse.** Ogni finestra deve prevedere da un momento del giorno diverso, altrimenti il
confronto fra modelli si riduce a un solo tipo di transizione. L'ora che conta e' quella del
**taglio** — il primo punto previsto — non quella in cui comincia il contesto.

**La tensione.** Il limite superiore di finestre disgiunte e' $\lfloor 525601/2112 \rfloor = 248$, e
tenendo conto dei buchi del logger circa $207$. Ma piu' finestre si chiedono, meno spazio resta per
scegliere *dove* metterle: a $230$ il margine dentro un blocco e' di $165$ minuti, cioe' due o tre
ore di taglio possibili, e l'istogramma orario esce sbilanciato. A $N = 150$ il margine e' di circa
$23$ ore e l'ora del taglio si puo' scegliere quasi liberamente. Il campionamento parte da $150$ per
questo; il parametro resta libero e la cella stampa l'istogramma che ne esce.

**Come si campiona.** Non a posizioni casuali: piazzare intervalli a caso su una retta e scartare
le sovrapposizioni si ferma alla costante di Rényi, circa il $74{,}8\%$ di riempimento. Si usa un
**campionamento sistematico con bilanciamento orario**: il file e' diviso in $N$ blocchi uguali,
i blocchi si percorrono in ordine casuale, e dentro ciascuno si sceglie fra le posizioni valide
quella la cui ora di taglio e' al momento **la meno rappresentata**. Il blocco garantisce la
copertura dell'anno, la scelta dentro il blocco appiana le ventiquattro ore, e la disgiunzione e'
per costruzione. Una seconda passata riempie gli spazi lasciati dai blocchi caduti su un buco.

**Quello che non si fa, e perche'.** Le ultime finestre fino a $207$ si otterrebbero piastrellando
i blocchi validi con finestre adiacenti, a passo esattamente $L = 2112$ minuti. Ma
$2112/1440 = 22/15$, quindi un campionatore cosi' ripete **lo stesso orario ogni quindici
finestre**: un pettine a passo fisso contro il ciclo diurno, cioe' esattamente l'errore che questo
progetto studia, commesso sul disegno sperimentale invece che nel modello. La cella stampa il
confronto fra i due passi, in orari distinti visitati prima di ripetersi, cosi' la scelta si vede
invece di doverla credere.

In [10]:
# finestra valida = tutti i suoi L_FIN campioni sono presenti. Test O(1) via somma cumulata.
_cs = np.concatenate([[0], np.cumsum(OK)])
VALIDA = (_cs[L_FIN:] - _cs[:-L_FIN]) == L_FIN        # VALIDA[s] per lo start s
print(f"posizioni di partenza valide: {VALIDA.sum():,} su {len(VALIDA):,}")
print(f"massimo teorico di finestre disgiunte: {T_TOT // L_FIN}")

def campiona_disgiunte(valida, n, L, seed, ctx=CTX):
    """N finestre disgiunte, distribuite sulle ventiquattro ore del giorno.

    Passata 1, sistematica e bilanciata. Il file e' diviso in n blocchi uguali e ogni blocco
    ospita una finestra. Il margine di manovra dentro un blocco e' (blocco - L) minuti, che
    copre due o tre ore di taglio diverse: fra le posizioni valide del blocco si sceglie quella
    la cui ORA DI TAGLIO e' al momento la meno rappresentata. Il blocco garantisce la copertura
    dell'anno, la scelta dentro il blocco appiana l'istogramma orario, e le finestre restano
    disgiunte perche' i blocchi lo sono.

    Passata 2, riempimento. Un blocco il cui margine cade dentro un buco del logger non produce
    niente. Si percorrono allora gli spazi fra finestre accettate e, dove ci sta una finestra
    intera, se ne piazza una con lo stesso criterio orario.

    L'ora e' quella del TAGLIO - il primo punto previsto - non quella dell'inizio del contesto:
    e' l'istante da cui ogni modello sta effettivamente prevedendo.
    """
    import bisect
    rng = np.random.default_rng(seed)
    T = len(valida)
    blocco = T // n
    if blocco < L:
        raise ValueError(f"blocco {blocco} < finestra {L}: N_FINESTRE troppo alto per questo file")

    conta = np.zeros(24, dtype=int)
    acc = []

    def _ora(s): return ((s + ctx) % 1440) // 60

    def _libera(s):
        i = bisect.bisect_left(acc, s)
        if i < len(acc) and acc[i] - s < L: return False
        if i > 0 and s - acc[i-1] < L: return False
        return True

    def _prendi(lo, hi, controlla_libera):
        """Fra le posizioni valide in [lo, hi], una dell'ora meno rappresentata. None se nessuna."""
        if hi < lo: return None
        amm = lo + np.flatnonzero(valida[lo:hi + 1])
        if controlla_libera and amm.size:
            amm = np.array([s for s in amm if _libera(int(s))], dtype=int)
        if amm.size == 0: return None
        ore = ((amm + ctx) % 1440) // 60
        c = conta[ore]
        migliori = amm[c == c.min()]
        return int(rng.choice(migliori))

    ordine = rng.permutation(n)              # i blocchi in ordine casuale, cosi' nessuna ora
    for k in ordine:                         # viene privilegiata dal solo scorrere del tempo
        s = _prendi(int(k)*blocco, min(int(k)*blocco + blocco - L, T - 1), False)
        if s is not None:
            bisect.insort(acc, s); conta[_ora(s)] += 1
    n1 = len(acc)

    while len(acc) < n:
        bordi = [-L] + acc + [T + L]
        spazi = [(bordi[i] + L, min(bordi[i+1] - L, T - 1)) for i in range(len(bordi) - 1)]
        spazi = [(a, b) for a, b in spazi if b >= a]
        aggiunte = 0
        for a, b in sorted(spazi, key=lambda ab: ab[1] - ab[0], reverse=True):
            if len(acc) >= n: break
            s = _prendi(a, b, True)
            if s is not None:
                bisect.insort(acc, s); conta[_ora(s)] += 1; aggiunte += 1
        if aggiunte == 0: break

    print(f"  passata sistematica: {n1}/{n}   riempimento degli spazi: +{len(acc)-n1}")
    if len(acc) < n:
        print(f"  il file non concede {n} finestre disgiunte valide: ottenute {len(acc)}")
    print(f"  finestre per ora del taglio: min {conta.min()}, max {conta.max()}, "
          f"ore mai visitate {(conta == 0).sum()}")
    return np.array(acc, dtype=int)


def _diagnosi_risonanza(passo, etichetta):
    """Quanti orari distinti visita un campionatore a passo fisso prima di ripetersi."""
    from math import gcd
    avanzo = passo % 1440
    distinti = 1440 // gcd(avanzo, 1440) if avanzo else 1
    print(f"  {etichetta:36s} passo {passo:6d} min -> avanzo {avanzo:4d} min/finestra, "
          f"{distinti:4d} orari distinti")
    return distinti

print("\nrisonanza col ciclo diurno (1440 min):")
_d_sis = _diagnosi_risonanza(len(VALIDA)//N_FINESTRE, "sistematico con jitter (usato)")
_d_pia = _diagnosi_risonanza(L_FIN, "piastrellatura adiacente (scartata)")
if _d_pia >= _d_sis:
    print("  ATTENZIONE: la piastrellatura non sarebbe peggiore, rivedere la scelta")

START = campiona_disgiunte(VALIDA, N_FINESTRE, L_FIN, SEED)
N = len(START)
assert np.all(np.diff(START) >= L_FIN), "le finestre si sovrappongono"
print(f"\nfinestre ottenute: {N}   disgiunte: si   passo minimo {np.diff(START).min()} >= {L_FIN}")

# contesti e veri. Con N di quest'ordine sta tutto in memoria: N x 2112 x 8 byte ~ 4 MB.
IDX  = START[:, None] + np.arange(L_FIN)[None, :]
CTXS = Y_ALL[IDX[:, :CTX]].astype(np.float32)          # [N, CTX]
YTRUE = Y_ALL[IDX[:, CTX:]].astype(np.float64)         # [N, PRED]
TCUT = RAW["Time"].to_numpy()[START + CTX]             # istante del primo punto previsto
assert np.isfinite(CTXS).all() and np.isfinite(YTRUE).all()

# regime, dall'elevazione solare sul solo TARGET (nessuna informazione dal futuro del canale)
_el = ELEV[IDX[:, CTX:]]
REGIME = np.where(_el.max(1) <= 0, "notte",
          np.where(_el.min(1) >= ELEV_GIORNO, "giorno", "transizione"))

FIN = pd.DataFrame({
    "finestra": np.arange(N), "start": START, "t_taglio": TCUT,
    "ora": pd.DatetimeIndex(TCUT).hour + pd.DatetimeIndex(TCUT).minute/60,
    "doy": pd.DatetimeIndex(TCUT).dayofyear, "regime": REGIME,
    "elev_max": _el.max(1).round(2), "y_media": YTRUE.mean(1).round(2),
    "y_max": YTRUE.max(1).round(2), "ctx_media": CTXS.mean(1).round(2)})
print("\nregimi:"); print(FIN.regime.value_counts().to_string())
print(f"copertura annuale: giorni {FIN.doy.min()} .. {FIN.doy.max()}, {FIN.doy.nunique()} distinti")
_orari = FIN.ora.astype(int).value_counts().reindex(range(24), fill_value=0)
print("finestre per ora del taglio, dalle 00 alle 23:")
print("  " + " ".join(f"{int(v):2d}" for v in _orari.to_numpy()))
print(f"  min {_orari.min()}, max {_orari.max()}, ore mai visitate {int((_orari == 0).sum())}")
assert _orari.min() >= 1, "c'e' un'ora del giorno da cui nessuna finestra prevede"
FIN.to_csv(f"{OUT_DIR}/finestre.csv", index=False)
display(FIN.head(8))

posizioni di partenza valide: 458,825 su 523,490
massimo teorico di finestre disgiunte: 248

risonanza col ciclo diurno (1440 min):
  sistematico con jitter (usato)       passo   3489 min -> avanzo  609 min/finestra,  480 orari distinti
  piastrellatura adiacente (scartata)  passo   2112 min -> avanzo  672 min/finestra,   15 orari distinti
  passata sistematica: 143/150   riempimento degli spazi: +7
  finestre per ora del taglio: min 6, max 7, ore mai visitate 0

finestre ottenute: 150   disgiunte: si   passo minimo 2116 >= 2112

regimi:
regime
notte          72
giorno         55
transizione    23
copertura annuale: giorni 2 .. 365, 150 distinti
finestre per ora del taglio, dalle 00 alle 23:
   6  6  6  7  7  7  6  7  7  6  6  6  6  6  6  7  6  6  6  6  6  6  6  6
  min 6, max 7, ore mai visitate 0


,finestra,start,t_taglio,ora,doy,regime,elev_max,y_media,y_max,ctx_media
0,0,208,2017-01-02 13:36:00,13.600000,2,giorno,19.73,188.19,397.0,70.239998
1,1,4142,2017-01-05 07:10:00,7.166667,5,transizione,0.61,1.53,15.0,58.320000
2,2,7180,2017-01-07 09:48:00,9.800000,7,giorno,18.47,265.27,324.0,69.160004
3,3,10889,2017-01-09 23:37:00,23.616667,9,notte,-64.28,0.28,2.0,77.690002
4,4,15004,2017-01-12 20:12:00,20.200000,12,notte,-32.80,0.23,1.0,21.889999
5,5,18460,2017-01-15 05:48:00,5.800000,15,notte,-11.83,0.02,1.0,55.509998
6,6,21423,2017-01-17 07:11:00,7.183333,17,transizione,1.45,1.06,8.0,69.129997
7,7,24578,2017-01-19 11:46:00,11.766667,19,giorno,23.99,406.50,419.0,88.550003


### Il manifesto

Ogni risultato salvato porta con se' le condizioni in cui e' stato prodotto. Se una ripresa
successiva cambia canale, contesto, numero di finestre o seme, le previsioni in cache non sono piu'
confrontabili e vanno rifatte: il controllo qui sotto se ne accorge invece di mescolare due run.

In [11]:
MANIFESTO = dict(canale=CANALE, ctx=CTX, pred=PRED, n_finestre=int(N), seed=SEED,
                 csv_sha=hashlib.sha256(Path(CSV_PATH).read_bytes()).hexdigest()[:16],
                 start_sha=hashlib.sha256(START.tobytes()).hexdigest()[:16])
_mp = Path(f"{OUT_DIR}/manifesto.json")
if _mp.exists():
    _vecchio = json.loads(_mp.read_text())
    _diff = {k: (_vecchio.get(k), v) for k, v in MANIFESTO.items() if _vecchio.get(k) != v}
    if _diff:
        print("MANIFESTO DIVERSO dalla cache presente. Le previsioni salvate NON sono riusabili:")
        for k, (a, b) in _diff.items(): print(f"   {k}: in cache {a!r}, ora {b!r}")
        print("   -> tutti i modelli verranno rifatti e i pred_*.npy sovrascritti")
        RIUSA_CACHE = False
        # Il manifesto va riscritto ORA, non a fine notebook: se la sessione cade a meta' loop,
        # su disco ci sarebbero previsioni nuove sotto un manifesto vecchio, e la ripresa
        # successiva le riuserebbe credendole valide. Con RIUSA_CACHE False tutti e sedici i
        # file vengono comunque riscritti, quindi allineare subito il manifesto e' corretto.
        _mp.write_text(json.dumps(MANIFESTO, indent=2))
    else:
        print("manifesto identico alla cache: le previsioni gia' calcolate verranno riusate")
        RIUSA_CACHE = True
else:
    _mp.write_text(json.dumps(MANIFESTO, indent=2)); RIUSA_CACHE = True
print(json.dumps(MANIFESTO, indent=2))

{
  "canale": "G_h",
  "ctx": 2048,
  "pred": 64,
  "n_finestre": 150,
  "seed": 42,
  "csv_sha": "7f9ff27fb276664b",
  "start_sha": "10527afabdff305f"
}


## 3, I modelli

I quindici sono `probe_lib.DELIVERABLE3_MODELS`, cioe' le geometrie del sweep che chiudono il
contesto ($480 \bmod S = 0$) al netto delle esclusioni dichiarate. E' la stessa popolazione su cui
gira l'analisi bayesiana, quindi la tabella e i modelli parlano delle stesse quindici cose.

Il sedicesimo e' `amazon/chronos-bolt-tiny` come rilasciato. Compare con l'etichetta
`base (pubblicato)` e non e' una geometria del sweep: e' il riferimento esterno.

In [12]:
if MODO_PROVA:
    GEOMS = [(8,8),(16,8),(16,12),(16,16),(24,8),(24,12),(24,16),(24,20),(24,24),
             (32,8),(32,12),(32,16),(32,20),(32,24),(32,32)]
    print("MODO PROVA: geometrie hardcoded, nessun checkpoint caricato")
else:
    import probe_lib as pl
    GEOMS = list(pl.DELIVERABLE3_MODELS)

assert RIF_INTERNO in GEOMS, f"{RIF_INTERNO} non e' fra le geometrie: manca il riferimento interno"
print(f"{len(GEOMS)} geometrie del sweep + 1 baseline pubblicato = {len(GEOMS)+1} righe")
print("  ", ", ".join(f"p{P}-s{S}" for P, S in GEOMS))

def tag(P, S): return f"p{P}-s{S}"
TAG_BASE = "base (pubblicato)"
TAG_RIF  = tag(*RIF_INTERNO)
RIGHE = [(TAG_BASE, None)] + [(tag(P, S), (P, S)) for P, S in GEOMS]

15 geometrie del sweep + 1 baseline pubblicato = 16 righe
   p8-s8, p16-s8, p16-s12, p16-s16, p24-s8, p24-s12, p24-s16, p24-s20, p24-s24, p32-s8, p32-s12, p32-s16, p32-s20, p32-s24, p32-s32


## 4, Le previsioni

Ogni modello riceve **le stesse** `N` finestre, nello stesso ordine, e restituisce tutti e nove i
quantili. Il file `pred_<tag>.npy` che ne esce ha forma `[N, 9, PRED]` e pesa meno di un megabyte,
quindi le previsioni si conservano tutte e le metriche si possono ricalcolare senza rifare i fit.

Un checkpoint per volta: si carica, si prevede, si libera la memoria della GPU. Le previsioni gia'
in cache si saltano.

In [13]:
QUANTILI = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
QI_MED   = QUANTILI.index(0.5)

def _predittore_finto(nome, seed):
    """Persistenza + deriva, con un errore che dipende dal tag. Solo per provare la pipeline."""
    rng = np.random.default_rng(abs(hash(nome)) % (2**31) + seed)
    scala = 1.0 + 0.10*rng.standard_normal()
    def f(X):
        ultimo = X[:, -1:].astype(np.float64)
        media  = X[:, -240:].mean(1, keepdims=True).astype(np.float64)
        base   = ultimo + (media - ultimo)*np.linspace(0, 1, PRED)[None, :]
        base   = base*scala + rng.normal(0, 8.0, size=(len(X), PRED))
        larg   = np.abs(base).mean()*0.25 + 1.0
        off    = np.array([-1.28,-0.84,-0.52,-0.25,0,0.25,0.52,0.84,1.28])[None,:,None]*larg
        return np.clip(base[:, None, :] + off, 0, None)
    return f

def _carica(geom, dev):
    """Pipeline e etichetta. geom None = il pubblicato, altrimenti il retrainato di quel (P,S)."""
    from chronos import BaseChronosPipeline
    if geom is None:
        return BaseChronosPipeline.from_pretrained(BASE_ID, device_map=dev), BASE_ID
    import model_loader as ml
    P, S = geom
    ck = ml.resolve_local_checkpoint(P, S)
    if ck is not None:
        return BaseChronosPipeline.from_pretrained(str(ck), device_map=dev), f"local {ck.name}"
    return (BaseChronosPipeline.from_pretrained(ml.SWEEP_REPO, subfolder=f"p{P}-s{S}-seed42",
                                                revision=ml.SWEEP_REVISION, device_map=dev),
            f"hub {ml.SWEEP_REPO[:28]}@{ml.SWEEP_REVISION[:8]}")

def prevedi_tutto(nome, geom):
    f = Path(f"{OUT_DIR}/pred_{nome.replace(' ','_').replace('(','').replace(')','')}.npy")
    if RIUSA_CACHE and f.exists():
        P_ = np.load(f)
        if P_.shape == (N, len(QUANTILI), PRED):
            print(f"  {nome:20s} da cache"); return P_
    t0 = time.time()
    if MODO_PROVA:
        out = _predittore_finto(nome, SEED)(CTXS); etichetta = "FINTO"
    else:
        import torch
        dev = DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
        pipe, etichetta = _carica(geom, dev)
        cfg = pipe.model.config.chronos_config
        if geom is not None and (int(cfg["input_patch_size"]), int(cfg["input_patch_stride"])) != geom:
            raise ValueError(f"{nome}: il checkpoint dichiara una geometria diversa")
        qs = list(cfg["quantiles"])
        sel = [qs.index(q) for q in QUANTILI] if all(q in qs for q in QUANTILI) else list(range(len(qs)))
        blocchi = []
        with torch.no_grad():
            for i in range(0, N, BATCH):
                xb = torch.tensor(CTXS[i:i+BATCH], device=dev)
                yb = pipe.predict(xb, prediction_length=PRED)      # [b, Q, PRED]
                blocchi.append(yb[:, sel, :].float().cpu().numpy())
        out = np.concatenate(blocchi, 0).astype(np.float64)
        del pipe
        if dev == "cuda": torch.cuda.empty_cache()
    assert out.shape == (N, len(QUANTILI), PRED), out.shape
    np.save(f, out)
    print(f"  {nome:20s} {etichetta:44s} {time.time()-t0:6.1f}s")
    return out

print(f"previsioni su {N} finestre, {len(RIGHE)} modelli"
      + ("   [MODO PROVA]" if MODO_PROVA else ""))
PRED_Q = {}
for nome, geom in RIGHE:
    PRED_Q[nome] = prevedi_tutto(nome, geom)
print("fatto.")

previsioni su 150 finestre, 16 modelli


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 34.6MB            

model.safetensors: downloading bytes:           |  0.00B            

...

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

p32-s32-seed42/model.safetensors: reconstructing file:   0%|          |  0.00B / 34.8MB            

p32-s32-seed42/model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

  p32-s32              hub federicosabbadini/chronos-bo@230ea282       7.0s
fatto.


## 5, Le metriche

**MAE, bias, varianza** sono calcolate sull'errore puntuale $e = \hat y_{0.5} - y$, con $\hat y_{0.5}$
la mediana dei nove quantili — Bolt non produce un punto, produce una distribuzione, e la mediana e'
il punto che le si estrae.

$$\mathrm{MAE} = \overline{|e|}, \qquad b = \bar e, \qquad V = \overline{(e - \bar e)^2},
\qquad \mathrm{RMSE}^2 = b^2 + V .$$

L'ultima e' un'identita', non un'approssimazione, e la cella la verifica numericamente. Va detto
che la decomposizione vale su MSE e **non** su MAE: "MAE = bias + varianza" non e' vera e non viene
scritta da nessuna parte qui.

**La weighted quantile loss** usa invece tutti e nove i quantili, ed e' la metrica con cui i lavori
su Chronos riportano i loro risultati:

$$\mathrm{WQL} = \frac{\sum_i \sum_q 2\big[\,q\,(y_i - \hat y_{i,q})\mathbf 1\{y_i \ge \hat y_{i,q}\}
+ (1-q)(\hat y_{i,q} - y_i)\mathbf 1\{y_i < \hat y_{i,q}\}\big]}{\sum_i |y_i|} .$$

Numeratore e denominatore si sommano **su tutte le finestre prima di dividere**. Il rapporto per
finestra sarebbe instabile: di notte $\sum|y_i| \to 0$ e il quoziente esplode su finestre che non
contengono informazione.

In [14]:
def metriche(pq, Y):
    """MAE, bias, varianza, RMSE e WQL da un blocco [N, Q, PRED] contro il vero [N, PRED]."""
    e = pq[:, QI_MED, :] - Y
    mae  = float(np.abs(e).mean()); bias = float(e.mean())
    var  = float(e.var(ddof=0));    mse  = float((e**2).mean())
    num = 0.0
    for k, q in enumerate(QUANTILI):
        d = Y - pq[:, k, :]
        num += 2.0*float(np.sum(np.where(d >= 0, q*d, (q - 1.0)*d)))
    den = float(np.abs(Y).sum())
    return dict(MAE=mae, bias=bias, Var=var, RMSE=float(np.sqrt(mse)),
                WQL=num/den if den > 0 else np.nan, _num=num, _den=den)

# verifica dell'identita' su ogni modello, non a campione
print("verifica RMSE^2 = bias^2 + Var")
_peggio = 0.0
for nome in PRED_Q:
    m = metriche(PRED_Q[nome], YTRUE)
    _peggio = max(_peggio, abs(m["RMSE"]**2 - (m["bias"]**2 + m["Var"]))/max(m["RMSE"]**2, 1e-12))
print(f"  scarto relativo massimo su {len(PRED_Q)} modelli: {_peggio:.3e}")
assert _peggio < 1e-9, "l'identita' non regge: c'e' un errore nel calcolo"

M = {nome: metriche(pq, YTRUE) for nome, pq in PRED_Q.items()}

verifica RMSE^2 = bias^2 + Var
  scarto relativo massimo su 16 modelli: 4.058e-16


## 6, Le tre tabelle

Un confronto solo non separa le due cose che differiscono fra questi sedici checkpoint — il corpus
su cui sono stati addestrati e la geometria con cui tokenizzano. Tre tabelle le separano.

| | confronto | cosa varia | cosa isola |
|---|---|---|---|
| **A** | `chronos-bolt-tiny` pubblicato contro `p16-s16` retrainato | **solo** il corpus | l'effetto del preaddestramento |
| **B** | il pubblicato contro tutti e quindici | corpus **e** geometria | il divario complessivo verso lo stato dell'arte |
| **C** | `p16-s16` retrainato contro gli altri quattordici | **solo** la geometria | la domanda del progetto |

**A** e' la tabella di controllo: stessa geometria $(16,16)$ da entrambe le parti, quindi tutto
quello che resta e' la differenza fra un corpus osservativo di ordine $10^7$ serie — che comprende
energia e meteo — e la sola mixture sintetica del sweep. Il numero che esce da A e' la scala
rispetto a cui vanno letti gli altri due.

**B** e' il confronto che si legge per primo ma dice meno di quanto sembri: ogni riga somma
l'effetto del corpus a quello della geometria, e senza A non c'e' modo di sapere quale dei due la
sta muovendo.

**C** e' l'unica delle tre che risponde alla domanda del progetto. Il riferimento e'
`p16-s16-seed42`: stessa mixture, stessi centomila step, stesso seme, geometria stock di Bolt. Il
pubblicato e' escluso perche' non appartiene a quella popolazione. Uno scarto il cui intervallo non
contiene lo zero e' una differenza attribuibile alla tokenizzazione, e nient'altro.

Le finestre sono disgiunte e distanti almeno $35$ ore, quindi la finestra **e'** l'unita'
indipendente e un bootstrap appaiato su di essa e' legittimo senza blocchi ulteriori. Appaiato
significa che a ogni ricampionamento tutti i modelli ricevono **le stesse** finestre: la differenza
fra due modelli non viene sporcata dal fatto che ne hanno viste di diverse. Il verdetto e' a tre
vie — l'intervallo sta tutto sotto zero, tutto sopra, oppure lo contiene e i due modelli non si
distinguono su queste finestre.

In [15]:
_rng_b = np.random.default_rng(SEED + 1)
BOOT_IDX = _rng_b.integers(0, N, size=(N_BOOT, N))     # gli stessi indici per tutti i modelli

def _mae_boot(pq):
    ae = np.abs(pq[:, QI_MED, :] - YTRUE).mean(1)       # MAE per finestra
    return ae[BOOT_IDX].mean(1)                          # [N_BOOT]

MAE_B = {nome: _mae_boot(pq) for nome, pq in PRED_Q.items()}
GEOM_DI = dict(RIGHE)

def _riga(nome, rif):
    g = GEOM_DI[nome]; m = M[nome]
    d = MAE_B[nome] - MAE_B[rif]
    lo, hi = float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))
    return dict(modello=nome, P=g[0] if g else np.nan, S=g[1] if g else np.nan,
                overlap=round(1 - g[1]/g[0], 3) if g else np.nan,
                MAE=m["MAE"], bias=m["bias"], Var=m["Var"], RMSE=m["RMSE"], WQL=m["WQL"],
                d_MAE=float(M[nome]["MAE"] - M[rif]["MAE"]), ic_lo=lo, ic_hi=hi,
                esito=("meglio" if hi < 0 else "peggio" if lo > 0 else "indistinguibile"))

def tabella(nomi, rif, titolo, file):
    righe = []
    for nome in nomi:
        r = _riga(nome, rif)
        if nome == rif:
            r.update(d_MAE=0.0, ic_lo=0.0, ic_hi=0.0, esito="(riferimento)")
        righe.append(r)
    T = pd.DataFrame(righe).sort_values("d_MAE").reset_index(drop=True)
    T.to_csv(f"{OUT_DIR}/{file}", index=False)
    print("\n" + "="*100); print(titolo); print("="*100)
    display(T.round(4))
    return T

_geom_tags = [t for t, g in RIGHE if g is not None]
_nota = "   [MODO PROVA - NUMERI NON REALI]" if MODO_PROVA else ""
print(f"canale {CANALE}   finestre {N}   bootstrap {N_BOOT}   intervalli al 95%{_nota}")
print("d_MAE < 0 = il modello sbaglia MENO del riferimento della sua tabella")

TAB_A = tabella([TAG_BASE, TAG_RIF], TAG_BASE,
                f"A. Il solo corpus: {BASE_ID} contro {TAG_RIF} retrainato  "
                "(stessa geometria da entrambe le parti)", "tab_A_corpus.csv")

TAB_B = tabella(_geom_tags, TAG_BASE,
                f"B. Corpus + geometria: i {len(_geom_tags)} retrainati contro il pubblicato  "
                "(le due cause sono sommate, non separate)", "tab_B_vs_pubblicato.csv")

TAB_C = tabella([t for t in _geom_tags if t != TAG_RIF] + [TAG_RIF], TAG_RIF,
                f"C. La sola geometria: gli altri {len(_geom_tags)-1} contro {TAG_RIF}  "
                "(stesso corpus, stessi step, stesso seme - il pubblicato e' escluso)",
                "tab_C_geometria.csv")

canale G_h   finestre 150   bootstrap 2000   intervalli al 95%
d_MAE < 0 = il modello sbaglia MENO del riferimento della sua tabella

A. Il solo corpus: amazon/chronos-bolt-tiny contro p16-s16 retrainato  (stessa geometria da entrambe le parti)


,modello,P,S,overlap,MAE,bias,Var,RMSE,WQL,d_MAE,ic_lo,ic_hi,esito
0,base (pubblicato),NaN,NaN,NaN,32.2826,7.6643,7584.4461,87.4253,1.5445,0.0000,0.000,0.0000,(riferimento)
1,p16-s16,16.0,16.0,0.0,42.8468,7.6479,7669.5558,87.9093,1.9347,10.5642,5.737,14.9133,peggio



B. Corpus + geometria: i 15 retrainati contro il pubblicato  (le due cause sono sommate, non separate)


,modello,P,S,overlap,MAE,bias,Var,RMSE,WQL,d_MAE,ic_lo,ic_hi,esito
0,p16-s8,16,8,0.500,37.9279,7.1648,6587.9733,81.4819,1.7388,5.6453,1.5898,9.2940,peggio
1,p32-s32,32,32,0.000,39.5948,7.2494,7978.0198,89.6135,1.8477,7.3122,3.3121,10.9551,peggio
2,p16-s16,16,16,0.000,42.8468,7.6479,7669.5558,87.9093,1.9347,10.5642,5.7370,14.9133,peggio
3,p24-s12,24,12,0.500,43.3600,6.3388,7801.4120,88.5528,1.9781,11.0774,5.6317,16.0131,peggio
4,p24-s24,24,24,0.000,46.3319,10.4153,7807.9026,88.9740,2.0396,14.0493,6.2522,21.6178,peggio
5,p8-s8,8,8,0.000,46.5032,3.8889,8364.0639,91.5379,2.0795,14.2206,8.0432,19.9903,peggio
6,p32-s12,32,12,0.625,47.9349,-16.9627,7845.1610,90.1826,2.1388,15.6523,8.2635,23.1447,peggio
7,p24-s8,24,8,0.667,50.5245,13.3531,8155.2475,91.2883,2.2751,18.2419,11.8297,24.8486,peggio
8,p32-s24,32,24,0.250,50.5904,3.6339,10110.8081,100.6182,2.2706,18.3078,12.5658,24.1786,peggio
9,p32-s16,32,16,0.500,50.7052,1.8837,8776.4587,93.7017,2.2613,18.4226,11.8990,24.4597,peggio



C. La sola geometria: gli altri 14 contro p16-s16  (stesso corpus, stessi step, stesso seme - il pubblicato e' escluso)


,modello,P,S,overlap,MAE,bias,Var,RMSE,WQL,d_MAE,ic_lo,ic_hi,esito
0,p16-s8,16,8,0.500,37.9279,7.1648,6587.9733,81.4819,1.7388,-4.9190,-8.1059,-2.0774,meglio
1,p32-s32,32,32,0.000,39.5948,7.2494,7978.0198,89.6135,1.8477,-3.2520,-5.4167,-1.1480,meglio
2,p16-s16,16,16,0.000,42.8468,7.6479,7669.5558,87.9093,1.9347,0.0000,0.0000,0.0000,(riferimento)
3,p24-s12,24,12,0.500,43.3600,6.3388,7801.4120,88.5528,1.9781,0.5132,-3.4512,4.2898,indistinguibile
4,p24-s24,24,24,0.000,46.3319,10.4153,7807.9026,88.9740,2.0396,3.4851,-2.2660,9.6965,indistinguibile
5,p8-s8,8,8,0.000,46.5032,3.8889,8364.0639,91.5379,2.0795,3.6564,-0.6144,8.1222,indistinguibile
6,p32-s12,32,12,0.625,47.9349,-16.9627,7845.1610,90.1826,2.1388,5.0881,-0.2387,10.3272,indistinguibile
7,p24-s8,24,8,0.667,50.5245,13.3531,8155.2475,91.2883,2.2751,7.6777,2.9335,12.6375,peggio
8,p32-s24,32,24,0.250,50.5904,3.6339,10110.8081,100.6182,2.2706,7.7436,4.1719,11.4504,peggio
9,p32-s16,32,16,0.500,50.7052,1.8837,8776.4587,93.7017,2.2613,7.8584,4.4132,11.3103,peggio


### Ripartizione per regime

Il $46\%$ circa dei target da 64 punti su questo canale e' interamente notturno, e una finestra
notturna la prevede bene chiunque: mediata dentro il totale, quella meta' comprime le differenze
fra modelli verso zero. La tabella principale resta su tutte le finestre — non si filtra in
silenzio — ma la ripartizione qui sotto dice quanta parte del MAE viene da dove.

In [16]:
_rows = []
for nome, pq in PRED_Q.items():
    ae = np.abs(pq[:, QI_MED, :] - YTRUE).mean(1)
    for reg in ["notte", "transizione", "giorno"]:
        sel = REGIME == reg
        if sel.sum() == 0: continue
        _rows.append(dict(modello=nome, regime=reg, finestre=int(sel.sum()),
                          MAE=float(ae[sel].mean()),
                          y_medio=float(YTRUE[sel].mean())))
REG = pd.DataFrame(_rows)
PIV = REG.pivot(index="modello", columns="regime", values="MAE").round(3)
REG.to_csv(f"{OUT_DIR}/per_regime.csv", index=False)
_cnt = {r: int((REGIME == r).sum()) for r in ["notte", "transizione", "giorno"]}
_ym  = {r: float(YTRUE[REGIME == r].mean()) for r in _cnt if (REGIME == r).any()}
print("MAE per regime (stesse finestre, stessa mediana)")
print("  finestre  " + "   ".join(f"{r}: {c}" for r, c in _cnt.items()))
print("  y medio   " + "   ".join(f"{r}: {v:.1f}" for r, v in _ym.items()))
display(PIV.sort_values("giorno"))

# MAE per orizzonte: costa nulla una volta che le previsioni sono in memoria, e dice se
# l'errore cresce liscio o a scalini lungo i 64 passi.
ORI = pd.DataFrame({nome: np.abs(pq[:, QI_MED, :] - YTRUE).mean(0)
                    for nome, pq in PRED_Q.items()})
ORI.index.name = "h"
ORI.to_csv(f"{OUT_DIR}/mae_per_orizzonte.csv")
print(f"\nMAE per orizzonte scritto in {OUT_DIR}/mae_per_orizzonte.csv  "
      f"(h=1: {ORI.iloc[0].min():.2f}-{ORI.iloc[0].max():.2f}, "
      f"h=64: {ORI.iloc[-1].min():.2f}-{ORI.iloc[-1].max():.2f})")

MAE per regime (stesse finestre, stessa mediana)
  finestre  notte: 72   transizione: 23   giorno: 55
  y medio   notte: 0.0   transizione: 46.9   giorno: 418.6


regime,giorno,notte,transizione
modello,,,
base (pubblicato),75.408,2.564,22.188
p16-s8,79.862,7.424,33.140
p32-s32,87.596,6.052,29.814
p16-s16,88.721,12.182,29.140
p24-s12,90.952,9.963,34.102
p24-s8,96.599,11.662,62.002
p24-s24,101.571,5.545,41.919
p8-s8,103.127,6.521,36.259
p32-s8,105.848,22.848,39.813



MAE per orizzonte scritto in _run/solar_bench/mae_per_orizzonte.csv  (h=1: 14.56-82.97, h=64: 41.12-143.53)


## 7, Come vanno lette

**A da' la scala.** Finche' non si sa quanto vale il solo cambio di corpus su queste finestre, uno
scarto di geometria non ha un metro. Se A vale molte unita' di MAE e gli scarti di C ne valgono
frazioni, la geometria e' un dettaglio rispetto ai dati di addestramento — che e' un risultato,
non un fallimento, e va detto.

**B non attribuisce.** Ogni riga somma le due cause. Se tutti e quindici stanno sopra il
pubblicato, la spiegazione piu' semplice e' il corpus, ed e' quella che A quantifica.

**C e' la tabella del progetto**, ed e' l'unica su cui ha senso discutere di tokenizzazione.

Tre cose che nessuna delle tre tabelle dice, e che conviene dichiarare prima che le chieda
qualcuno. E' un test su **un sito e un anno**: la popolazione e' una, e nulla qui la generalizza ad
altre serie. E' un test **zero-shot** per i quindici, che la telemetria non l'hanno mai vista, ma
non e' detto che lo sia per il pubblicato, il cui corpus di preaddestramento comprende serie
energetiche e meteo e potrebbe contenere questo dataset o suoi parenti — un'altra ragione per non
leggere B come una classifica di merito. E le finestre sono centocinquanta: bastano per uno scarto
grande, non per uno piccolo, e l'intervallo bootstrap e' li' per dire quale dei due si sta
guardando. Il verdetto *indistinguibile* non significa "uguali": significa che queste finestre non
bastano a separarli.

In [17]:
print("="*78); print("RIEPILOGO"); print("="*78)

_a = TAB_A[TAB_A.modello == TAG_RIF].iloc[0]
print(f"A, solo corpus      {TAG_RIF} contro il pubblicato: "
      f"{_a.d_MAE:+.3f} MAE  [{_a.ic_lo:+.3f}, {_a.ic_hi:+.3f}]   {_a.esito}")

_mb = TAB_B[TAB_B.esito == "meglio"]; _pb = TAB_B[TAB_B.esito == "peggio"]
print(f"B, corpus+geometria  meglio del pubblicato {len(_mb)}/{len(TAB_B)}, "
      f"peggio {len(_pb)}/{len(TAB_B)}, indistinguibili "
      f"{len(TAB_B)-len(_mb)-len(_pb)}/{len(TAB_B)}")
if len(_mb): print("   meglio: " + ", ".join(_mb.modello))

_c = TAB_C[TAB_C.modello != TAG_RIF]
_mc = _c[_c.esito == "meglio"]; _pc = _c[_c.esito == "peggio"]
print(f"C, sola geometria    meglio di {TAG_RIF} {len(_mc)}/{len(_c)}, "
      f"peggio {len(_pc)}/{len(_c)}, indistinguibili {len(_c)-len(_mc)-len(_pc)}/{len(_c)}")
if len(_mc): print("   meglio: " + ", ".join(_mc.modello))
if len(_pc): print("   peggio: " + ", ".join(_pc.modello))

print(f"\nscala di riferimento: l'effetto del solo corpus vale {abs(_a.d_MAE):.3f} di MAE.")
print("Uno scarto di geometria molto piu' piccolo di questo e' un dettaglio; uno")
print("confrontabile o maggiore e' la cosa che il progetto sta cercando.")
print(f"\nscritti in {OUT_DIR}/: tab_A_corpus.csv, tab_B_vs_pubblicato.csv, tab_C_geometria.csv,")
print(f"                       per_regime.csv, mae_per_orizzonte.csv, finestre.csv, pred_*.npy")
if MODO_PROVA:
    print("\n*** MODO PROVA ATTIVO: i numeri vengono da un predittore finto, non dai modelli. ***")

RIEPILOGO
A, solo corpus      p16-s16 contro il pubblicato: +10.564 MAE  [+5.737, +14.913]   peggio
B, corpus+geometria  meglio del pubblicato 0/15, peggio 15/15, indistinguibili 0/15
C, sola geometria    meglio di p16-s16 2/14, peggio 8/14, indistinguibili 4/14
   meglio: p16-s8, p32-s32
   peggio: p24-s8, p32-s24, p32-s16, p24-s20, p24-s16, p32-s8, p16-s12, p32-s20

scala di riferimento: l'effetto del solo corpus vale 10.564 di MAE.
Uno scarto di geometria molto piu' piccolo di questo e' un dettaglio; uno
confrontabile o maggiore e' la cosa che il progetto sta cercando.

scritti in _run/solar_bench/: tab_A_corpus.csv, tab_B_vs_pubblicato.csv, tab_C_geometria.csv,
                       per_regime.csv, mae_per_orizzonte.csv, finestre.csv, pred_*.npy
